# 01. ChEMBL 데이터 수집

**목표**: ChEMBL에서 ADC 페이로드 후보 화합물의 세포독성(IC50) 데이터 수집  
**전략 A**: cytotoxicity 키워드로 assay 검색  
**전략 B**: 알려진 ADC 페이로드 계열 직접 검색 (MMAE, DM1, SN-38 등)  


## 0. 라이브러리 로드

In [ ]:
import sys
import os
import pandas as pd
from tqdm.notebook import tqdm
from chembl_webresource_client.new_client import new_client

# 상위 폴더의 src 모듈 사용
sys.path.append(os.path.join(os.getcwd(), '..'))

activity = new_client.activity
assay    = new_client.assay
molecule = new_client.molecule

print('라이브러리 로드 완료')

## 1. 탐색 — 데이터가 얼마나 있는지 먼저 확인

본격 수집 전에 각 전략별로 데이터가 몇 개 나오는지 빠르게 확인한다.

In [ ]:
# ── 전략 A 탐색: cytotoxicity assay 몇 개? ──
assays_cyto = list(
    assay.filter(
        description__icontains='cytotoxicity',
        assay_type='B'
    ).only(['assay_chembl_id', 'description'])[:5]  # 샘플 5개만
)

print(f'cytotoxicity assay 샘플 ({len(assays_cyto)}개):')
for a in assays_cyto:
    print(f"  {a['assay_chembl_id']} | {a['description'][:80]}")

In [ ]:
# ── 전략 B 탐색: MMAE 검색 샘플 ──
mols_mmae = list(
    molecule.filter(
        pref_name__icontains='auristatin'
    ).only(['molecule_chembl_id', 'pref_name'])[:10]
)

print(f'auristatin 계열 화합물 ({len(mols_mmae)}개):')
for m in mols_mmae:
    print(f"  {m['molecule_chembl_id']} | {m['pref_name']}")

In [ ]:
# ── 특정 화합물의 IC50 데이터 샘플 확인 ──
if mols_mmae:
    sample_id = mols_mmae[0]['molecule_chembl_id']
    acts = list(
        activity.filter(
            molecule_chembl_id=sample_id,
            standard_type='IC50',
            standard_units='nM',
        ).only(['standard_value', 'assay_chembl_id', 'canonical_smiles'])[:5]
    )
    print(f"{sample_id} IC50 데이터 샘플 ({len(acts)}개):")
    for act in acts:
        print(f"  IC50={act.get('standard_value')} nM | assay={act.get('assay_chembl_id')}")

## 2. 본격 수집

> ⏳ API 호출이라 시간이 걸릴 수 있어. `max_assays` 값으로 속도 조절 가능.

In [ ]:
# ── 전략 A: cytotoxicity assay 기반 수집 ──
ADC_PAYLOAD_KEYWORDS = [
    'monomethyl auristatin', 'auristatin',
    'maytansine', 'emtansine',
    'calicheamicin',
    'SN-38',
    'dxd',
    'pyrrolobenzodiazepine',
    'duocarmycin',
    'camptothecin',
]

MAX_ASSAYS = 30  # 너무 많으면 시간 오래 걸림. 처음엔 30 추천

print('[전략 A] cytotoxicity assay 검색')
assays_list = list(
    assay.filter(
        description__icontains='cytotoxicity',
        assay_type='B'
    ).only(['assay_chembl_id', 'description'])[:MAX_ASSAYS]
)
print(f'  → {len(assays_list)}개 assay 발견')

records_a = []
for a in tqdm(assays_list, desc='assay 순회'):
    acts = list(
        activity.filter(
            assay_chembl_id=a['assay_chembl_id'],
            standard_type='IC50',
            standard_units='nM',
        ).only(['molecule_chembl_id', 'canonical_smiles',
                'standard_value', 'assay_chembl_id'])
    )
    for act in acts:
        if act.get('standard_value') and act.get('canonical_smiles'):
            records_a.append({
                'molecule_chembl_id': act['molecule_chembl_id'],
                'smiles':             act['canonical_smiles'],
                'ic50_nM':            float(act['standard_value']),
                'assay_id':           act['assay_chembl_id'],
                'assay_description':  a['description'],
                'source':             'assay_keyword',
            })

df_a = pd.DataFrame(records_a)
print(f'  → {len(df_a)}개 수집 완료')
df_a.head(3)

In [ ]:
# ── 전략 B: ADC 페이로드 계열 직접 검색 ──
print('[전략 B] ADC 페이로드 계열 검색')
records_b = []

for kw in tqdm(ADC_PAYLOAD_KEYWORDS, desc='페이로드 키워드'):
    mols = list(
        molecule.filter(
            pref_name__icontains=kw
        ).only(['molecule_chembl_id', 'pref_name', 'molecule_structures'])
    )
    for mol in mols:
        chembl_id = mol.get('molecule_chembl_id')
        smiles = (mol.get('molecule_structures') or {}).get('canonical_smiles')
        if not smiles:
            continue
        acts = list(
            activity.filter(
                molecule_chembl_id=chembl_id,
                standard_type='IC50',
                standard_units='nM',
            ).only(['standard_value', 'assay_chembl_id'])
        )
        for act in acts:
            if act.get('standard_value'):
                records_b.append({
                    'molecule_chembl_id': chembl_id,
                    'smiles':             smiles,
                    'ic50_nM':            float(act['standard_value']),
                    'assay_id':           act.get('assay_chembl_id', ''),
                    'assay_description':  mol.get('pref_name', kw),
                    'source':             'payload_keyword',
                })

df_b = pd.DataFrame(records_b)
print(f'  → {len(df_b)}개 수집 완료')
df_b.head(3)

## 3. 합치기 + 정제

In [ ]:
# 두 전략 합치기
df_raw = pd.concat([df_a, df_b], ignore_index=True)
print(f'합친 원본 행 수: {len(df_raw)}')
print(f'source 분포:\n{df_raw["source"].value_counts()}')

In [ ]:
# 정제
df = df_raw.copy()

# 1) 결측 제거
df = df.dropna(subset=['smiles', 'ic50_nM'])

# 2) 이상값 제거 (0 이하, 1M nM 초과)
df = df[(df['ic50_nM'] > 0) & (df['ic50_nM'] <= 1_000_000)]

# 3) 중복 제거
df = df.drop_duplicates(subset=['molecule_chembl_id', 'assay_id'])

# 4) 이진 라벨링 (100 nM 기준)
df['label'] = (df['ic50_nM'] < 100).astype(int)

print(f'정제 후 행 수: {len(df)}')
print(f'\n라벨 분포:')
print(df['label'].value_counts())
print(f'\n클래스 비율: {df["label"].mean():.2%} 가 고독성(1)')

In [ ]:
# IC50 분포 빠르게 확인
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 원래 스케일
axes[0].hist(df['ic50_nM'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(100, color='red', linestyle='--', label='기준: 100 nM')
axes[0].set_xlabel('IC50 (nM)')
axes[0].set_ylabel('count')
axes[0].set_title('IC50 분포')
axes[0].legend()

# log 스케일
axes[1].hist(np.log10(df['ic50_nM']), bins=50, color='salmon', edgecolor='white')
axes[1].axvline(np.log10(100), color='red', linestyle='--', label='log10(100) = 2')
axes[1].set_xlabel('log10(IC50)')
axes[1].set_ylabel('count')
axes[1].set_title('IC50 분포 (log scale)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/ic50_distribution.png', dpi=150)
plt.show()
print('분포 그래프 저장 완료')

## 4. 저장

In [ ]:
import os
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# 원본 저장
df_raw.to_csv('../data/raw/chembl_raw.csv', index=False)
print(f'원본 저장: data/raw/chembl_raw.csv ({len(df_raw)}행)')

# 정제본 저장
df.to_csv('../data/processed/chembl_clean.csv', index=False)
print(f'정제본 저장: data/processed/chembl_clean.csv ({len(df)}행)')

print('\n최종 데이터 미리보기:')
df[['molecule_chembl_id', 'smiles', 'ic50_nM', 'label', 'source']].head()

## 5. 수집 결과 요약

> 아래 셀을 실행하면 2일차 작업 완료 여부를 체크할 수 있어.

In [ ]:
print('=' * 50)
print('2일차 데이터 수집 결과 요약')
print('=' * 50)
print(f'총 화합물 수        : {df["molecule_chembl_id"].nunique()}개')
print(f'총 데이터 포인트    : {len(df)}개')
print(f'고독성 (IC50<100nM) : {df["label"].sum()}개 ({df["label"].mean():.1%})')
print(f'저독성 (IC50≥100nM) : {(df["label"]==0).sum()}개 ({1-df["label"].mean():.1%})')
print(f'IC50 중앙값         : {df["ic50_nM"].median():.1f} nM')
print('=' * 50)

if len(df) < 100:
    print('⚠️  데이터가 적어. MAX_ASSAYS 늘리거나 키워드 추가 검토')
elif df['label'].mean() < 0.1 or df['label'].mean() > 0.9:
    print('⚠️  클래스 불균형 심함. 3일차에 라벨링 기준값 조정 필요')
else:
    print('✅ 데이터 충분, 3일차 라벨링 기준 검토로 넘어가자!')